the code block below takes a skeletonized mask image and find out 3 corner points:
- folder below has skeletonized masks for the patient 0401: "/Users/ruhisharmin/Library/CloudStorage/Box-Box/aether.lab/projects/Echo.Cardio/autosegmentation/Data/CAMUS/validation_pred_mask_2_Folder_Skeletonized/patient0401"
- code below runs in a loop and saves the 3 (X,Y) cooridnate pairs in both a .hdf5 file and a .csv file format where each row corresponds to the points of each tiemstep. Also, visulaize the points overlaid on the skeletonized contour.
1. Point Detection: Detects the 3 key cardiac points (Top, End1, End2) on each skeletonized contour
2. Data Storage: Saves results in a CSV file with columns for the timestep and each point's X,Y coordinates
3. Visualization: Generates individual visualizations for each timestep showing the points on the contour
4. Organization: Saves visualizations in a separate subfolder 

- If needed to process other patients, modify the input and output directory paths in the __main__ section of the script.
- the required libraries:
   pip install numpy opencv-python matplotlib pandas h5py (or !pip)
- install plotly for the interactive visualization:
   pip install plotly
- output will be the interactive HTML viewer, which allows to see the exact coordinates when hover over each point. This file can be opened in any web browser and doesn't require Python to use once it's created.

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
import pandas as pd
import h5py
import re

def detect_cardiac_points(image_path):
    """
    Detects the three key points on a cardiac skeletonized contour.
    
    Parameters:
    -----------
    image_path : str
        Path to the skeletonized image file
        
    Returns:
    --------
    tuple
        (point3, end1, end2) coordinates as tuples
    """
    # Load image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image from {image_path}")
    
    # Ensure binary image
    if img.max() > 1:
        _, binary = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
    else:
        binary = img * 255
    
    # Get all white pixels
    y_coords, x_coords = np.where(binary > 0)
    
    if len(x_coords) < 3:
        raise ValueError("Not enough points in the image")
    
    # STEP 1: Find the top point (minimum y-coordinate)
    # First, filter out any points that might be noise or outside the main contour
    # We'll focus on the central region of the contour by x-coordinate
    x_min, x_max = np.min(x_coords), np.max(x_coords)
    x_range = x_max - x_min
    x_center = (x_min + x_max) / 2
    
    # Consider points within the central 70% of the x-range
    central_x_min = x_center - 0.35 * x_range
    central_x_max = x_center + 0.35 * x_range
    
    central_mask = (x_coords >= central_x_min) & (x_coords <= central_x_max)
    central_y = y_coords[central_mask]
    central_x = x_coords[central_mask]
    
    if len(central_y) > 0:
        # Find the minimum y-coordinate within this central region
        central_min_y_idx = np.argmin(central_y)
        point3 = (int(central_x[central_min_y_idx]), int(central_y[central_min_y_idx]))
    else:
        # Fallback to original method if no central points found
        min_y_idx = np.argmin(y_coords)
        point3 = (int(x_coords[min_y_idx]), int(y_coords[min_y_idx]))
    
    # STEP 2: Find both endpoints
    # Calculate median x to separate left from right
    x_median = np.median(x_coords)
    
    # Find points in the bottom region (bottom 30%)
    y_max = np.max(y_coords)
    y_min = np.min(y_coords)
    bottom_threshold = y_min + 0.7 * (y_max - y_min)
    
    # Get bottom points
    bottom_mask = y_coords > bottom_threshold
    bottom_x = x_coords[bottom_mask]
    bottom_y = y_coords[bottom_mask]
    
    # If we don't find enough points in the bottom region, lower the threshold
    if len(bottom_x) < 2:
        bottom_threshold = y_min + 0.5 * (y_max - y_min)
        bottom_mask = y_coords > bottom_threshold
        bottom_x = x_coords[bottom_mask]
        bottom_y = y_coords[bottom_mask]
    
    # Split bottom points into left and right sides
    left_mask = bottom_x < x_median
    right_mask = bottom_x >= x_median
    
    # Find the bottommost point on the left side
    if np.any(left_mask):
        left_y = bottom_y[left_mask]
        left_x = bottom_x[left_mask]
        left_bottom_idx = np.argmax(left_y)
        end1 = (int(left_x[left_bottom_idx]), int(left_y[left_bottom_idx]))
    else:
        # Fallback: use leftmost point
        left_idx = np.argmin(x_coords)
        end1 = (int(x_coords[left_idx]), int(y_coords[left_idx]))
    
    # Find the bottommost point on the right side
    if np.any(right_mask):
        right_y = bottom_y[right_mask]
        right_x = bottom_x[right_mask]
        right_bottom_idx = np.argmax(right_y)
        end2 = (int(right_x[right_bottom_idx]), int(right_y[right_bottom_idx]))
    else:
        # Fallback: use rightmost point
        right_idx = np.argmax(x_coords)
        end2 = (int(x_coords[right_idx]), int(y_coords[right_idx]))
    
    return point3, end1, end2

def visualize_cardiac_points(image_path, points, output_path=None):
    """
    Visualizes the three key points on the cardiac contour.
    
    Parameters:
    -----------
    image_path : str
        Path to the skeletonized image file
    points : tuple
        (point3, end1, end2) coordinates as tuples
    output_path : str, optional
        Path to save the visualization result
    """
    # Unpack points
    point3, end1, end2 = points
    
    # Load image
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Could not read image from {image_path}")
    
    # Ensure binary image
    if img.max() > 1:
        _, binary = cv2.threshold(img, 127, 255, cv2.THRESH_BINARY)
    else:
        binary = img * 255
    
    # Create a matplotlib figure for interactive visualization
    plt.figure(figsize=(10, 8))
    
    # Display the binary image in grayscale
    plt.imshow(binary, cmap='gray')
    
    # Plot the points with distinct colors and markers
    plt.plot(point3[0], point3[1], 'yo', markersize=10, label=f'Top ({point3[0]}, {point3[1]})')
    plt.plot(end1[0], end1[1], 'bo', markersize=10, label=f'End1 ({end1[0]}, {end1[1]})')
    plt.plot(end2[0], end2[1], 'ro', markersize=10, label=f'End2 ({end2[0]}, {end2[1]})')
    
    # Add labels with coordinate information
    plt.annotate(f'Top: ({point3[0]}, {point3[1]})', 
                 xy=point3, 
                 xytext=(point3[0] + 15, point3[1] - 15),
                 color='yellow',
                 fontsize=10,
                 bbox=dict(boxstyle="round,pad=0.3", fc="black", alpha=0.7))
    
    plt.annotate(f'End1: ({end1[0]}, {end1[1]})', 
                 xy=end1, 
                 xytext=(end1[0] + 15, end1[1]),
                 color='cyan',
                 fontsize=10,
                 bbox=dict(boxstyle="round,pad=0.3", fc="black", alpha=0.7))
    
    plt.annotate(f'End2: ({end2[0]}, {end2[1]})', 
                 xy=end2, 
                 xytext=(end2[0] + 15, end2[1]),
                 color='red',
                 fontsize=10,
                 bbox=dict(boxstyle="round,pad=0.3", fc="black", alpha=0.7))
    
    # Configure the plot
    plt.title(f'Cardiac Points - {os.path.basename(image_path)}')
    plt.grid(True, alpha=0.3)
    plt.legend(loc='upper right')
    
    # Show axes and enable hover/cursor coordinate display
    plt.axis('on')
    
    # Extract only the file name from the full path for cleaner display
    base_filename = os.path.basename(image_path)
    
    # Create a tight layout
    plt.tight_layout()
    
    # Save the visualization if requested
    if output_path:
        plt.savefig(output_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        # Also create a simple OpenCV version for thumbnail/preview purposes
        vis_img = np.zeros((binary.shape[0], binary.shape[1], 3), dtype=np.uint8)
        vis_img[binary > 0] = [255, 255, 255]
        
        # Draw points
        cv2.circle(vis_img, point3, 8, (0, 255, 255), -1)  # Top (yellow)
        cv2.circle(vis_img, end1, 8, (255, 0, 0), -1)      # End1 (blue)
        cv2.circle(vis_img, end2, 8, (0, 0, 255), -1)      # End2 (red)
        
        # Save OpenCV version as a thumbnail
        thumbnail_path = output_path.replace('.png', '_thumb.png')
        cv2.imwrite(thumbnail_path, vis_img)
        
        return vis_img
    else:
        plt.show()
        return None

def process_patient_directory(input_dir, output_dir):
    """
    Process all skeletonized images in a patient directory,
    extract cardiac points, and save the results.
    
    Parameters:
    -----------
    input_dir : str
        Path to the directory containing skeletonized images
    output_dir : str
        Path to the directory to save results
    """
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    vis_dir = os.path.join(output_dir, "visualizations")
    os.makedirs(vis_dir, exist_ok=True)
    
    # List all skeletonized images
    file_list = sorted([f for f in os.listdir(input_dir) if f.endswith('_Skeletonized.png')])
    
    if not file_list:
        print(f"No skeletonized images found in {input_dir}")
        return
    
    # Prepare data structures for results
    results_df = pd.DataFrame(columns=[
        'Timestep', 
        'Top_X', 'Top_Y', 
        'End1_X', 'End1_Y', 
        'End2_X', 'End2_Y'
    ])
    
    # Process each image
    for file_name in file_list:
        print(f"Processing {file_name}...")
        
        # Extract timestep number
        match = re.search(r'tst_(\d+)_pred_mask_2_Skeletonized\.png', file_name)
        if not match:
            print(f"  Skipping file with unexpected format: {file_name}")
            continue
            
        timestep = match.group(1)
        
        # Process the image
        try:
            image_path = os.path.join(input_dir, file_name)
            points = detect_cardiac_points(image_path)
            point3, end1, end2 = points
            
            # Add to dataframe
            results_df = pd.concat([results_df, pd.DataFrame({
                'Timestep': [timestep],
                'Top_X': [point3[0]], 'Top_Y': [point3[1]],
                'End1_X': [end1[0]], 'End1_Y': [end1[1]],
                'End2_X': [end2[0]], 'End2_Y': [end2[1]]
            })], ignore_index=True)
            
            # Create visualization
            vis_path = os.path.join(vis_dir, f"tst_{timestep}_points.png")
            visualize_cardiac_points(image_path, points, vis_path)
            
            print(f"  Points extracted: Top={point3}, End1={end1}, End2={end2}")
            
        except Exception as e:
            print(f"  Error processing {file_name}: {e}")
    
    # Sort results by timestep
    results_df['Timestep'] = results_df['Timestep'].astype(int)
    results_df = results_df.sort_values('Timestep')
    
    # Save results to CSV
    csv_path = os.path.join(output_dir, "cardiac_points.csv")
    results_df.to_csv(csv_path, index=False)
    print(f"Results saved to CSV: {csv_path}")
    
    # Save results to HDF5
    h5_path = os.path.join(output_dir, "cardiac_points.h5")
    with h5py.File(h5_path, 'w') as h5f:
        # Create dataset for each point type
        timesteps = results_df['Timestep'].values.astype(np.int32)
        h5f.create_dataset('Timestep', data=timesteps)
        
        # Create datasets for coordinates with explicit dtype (float32)
        # Top points
        top_x = results_df['Top_X'].values.astype(np.float32)
        top_y = results_df['Top_Y'].values.astype(np.float32)
        h5f.create_dataset('Top_X', data=top_x)
        h5f.create_dataset('Top_Y', data=top_y)
        
        # End1 points
        end1_x = results_df['End1_X'].values.astype(np.float32)
        end1_y = results_df['End1_Y'].values.astype(np.float32)
        h5f.create_dataset('End1_X', data=end1_x)
        h5f.create_dataset('End1_Y', data=end1_y)
        
        # End2 points
        end2_x = results_df['End2_X'].values.astype(np.float32)
        end2_y = results_df['End2_Y'].values.astype(np.float32)
        h5f.create_dataset('End2_X', data=end2_x)
        h5f.create_dataset('End2_Y', data=end2_y)
        
        # Add attributes
        h5f.attrs['PatientID'] = 'patient0401'
        h5f.attrs['NumTimesteps'] = len(timesteps)
        h5f.attrs['CreationDate'] = np.string_(pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S'))
    
    print(f"Results saved to HDF5: {h5_path}")
    
    # Create a summary visualization
    create_summary_visualization(results_df, input_dir, os.path.join(output_dir, "points_summary.png"))
    
    return results_df

def create_summary_visualization(results_df, input_dir, output_path):
    """
    Creates a summary visualization showing points from all timesteps.
    
    Parameters:
    -----------
    results_df : pandas.DataFrame
        DataFrame containing the cardiac points data
    input_dir : str
        Path to the directory containing skeletonized images
    output_path : str
        Path to save the summary visualization
    """
    # Find the dimensions of the images
    sample_file = os.path.join(input_dir, f"tst_{results_df['Timestep'].iloc[0]:04d}_pred_mask_2_Skeletonized.png")
    sample_img = cv2.imread(sample_file, cv2.IMREAD_GRAYSCALE)
    if sample_img is None:
        print(f"Could not create summary visualization: sample image not found")
        return
    
    height, width = sample_img.shape
    
    # Create a figure with subplots arranged in a grid
    num_images = len(results_df)
    rows = int(np.ceil(np.sqrt(num_images)))
    cols = int(np.ceil(num_images / rows))
    
    plt.figure(figsize=(cols * 4, rows * 4))
    
    for i, (_, row) in enumerate(results_df.iterrows()):
        timestep = int(row['Timestep'])
        
        # Load image
        image_path = os.path.join(input_dir, f"tst_{timestep:04d}_pred_mask_2_Skeletonized.png")
        img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        
        if img is None:
            continue
        
        # Extract points
        point3 = (int(row['Top_X']), int(row['Top_Y']))
        end1 = (int(row['End1_X']), int(row['End1_Y']))
        end2 = (int(row['End2_X']), int(row['End2_Y']))
        
        # Add to subplot with axes
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(img, cmap='gray')
        
        # Plot points
        ax.plot(point3[0], point3[1], 'yo', markersize=8)
        ax.plot(end1[0], end1[1], 'bo', markersize=8)
        ax.plot(end2[0], end2[1], 'ro', markersize=8)
        
        # Show coordinates in a compact format
        ax.set_title(f"Timestep {timestep}\nT({point3[0]},{point3[1]}) E1({end1[0]},{end1[1]}) E2({end2[0]},{end2[1]})", fontsize=8)
        
        # Show axes but keep them minimal
        ax.set_xticks([0, width//2, width])
        ax.set_yticks([0, height//2, height])
        ax.tick_params(axis='both', which='both', labelsize=6)
        ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    
    # Also create an interactive HTML visualization
    html_path = output_path.replace('.png', '.html')
    try:
        from matplotlib.backends.backend_html import FigureManagerHTML
        plt.savefig(html_path, format='html')
        print(f"Interactive HTML summary saved to: {html_path}")
    except:
        print("Could not create interactive HTML summary (matplotlib HTML backend not available)")
    
    plt.close()
    
    print(f"Summary visualization saved to: {output_path}")
    
    # Also create a standalone interactive visualization with plotly
    try:
        import plotly.graph_objects as go
        from plotly.subplots import make_subplots
        
        fig = make_subplots(rows=rows, cols=cols, subplot_titles=[f"Timestep {t}" for t in results_df['Timestep']])
        
        for i, (_, row) in enumerate(results_df.iterrows()):
            timestep = int(row['Timestep'])
            r = (i // cols) + 1
            c = (i % cols) + 1
            
            # Load image
            image_path = os.path.join(input_dir, f"tst_{timestep:04d}_pred_mask_2_Skeletonized.png")
            img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                continue
                
            # Extract points
            point3 = (int(row['Top_X']), int(row['Top_Y']))
            end1 = (int(row['End1_X']), int(row['End1_Y']))
            end2 = (int(row['End2_X']), int(row['End2_Y']))
            
            # Add image
            fig.add_trace(
                go.Heatmap(
                    z=img,
                    colorscale='gray',
                    showscale=False,
                    hoverinfo='none'
                ),
                row=r, col=c
            )
            
            # Add points
            fig.add_trace(
                go.Scatter(
                    x=[point3[0], end1[0], end2[0]],
                    y=[point3[1], end1[1], end2[1]],
                    mode='markers',
                    marker=dict(
                        size=10,
                        color=['yellow', 'blue', 'red'],
                        line=dict(width=2, color='black')
                    ),
                    name='Points',
                    hovertext=[
                        f'Top: ({point3[0]}, {point3[1]})',
                        f'End1: ({end1[0]}, {end1[1]})',
                        f'End2: ({end2[0]}, {end2[1]})'
                    ],
                    hoverinfo='text'
                ),
                row=r, col=c
            )
            
        # Set layout
        fig.update_layout(
            title_text="Cardiac Points Summary (Interactive)",
            height=rows * 400,
            width=cols * 400,
            showlegend=False
        )
        
        # Save as interactive HTML
        plotly_html_path = output_path.replace('.png', '_interactive.html')
        fig.write_html(plotly_html_path)
        print(f"Interactive plotly visualization saved to: {plotly_html_path}")
    except Exception as e:
        print(f"Could not create plotly interactive visualization: {e}")

def create_interactive_viewer(results_df, input_dir, output_path):
    """
    Creates an HTML file with an interactive viewer for cardiac points.
    This allows viewing coordinates on hover.
    
    Parameters:
    -----------
    results_df : pandas.DataFrame
        DataFrame containing the cardiac points data
    input_dir : str
        Path to the directory containing skeletonized images
    output_path : str
        Path to save the interactive viewer HTML file
    """
    try:
        import base64
        from io import BytesIO
        
        # HTML template with embedded JavaScript for interactivity
        html_template = """
        <!DOCTYPE html>
        <html>
        <head>
            <title>Cardiac Points Interactive Viewer</title>
            <style>
                body { font-family: Arial, sans-serif; margin: 20px; }
                .container { display: flex; flex-wrap: wrap; }
                .image-container { 
                    margin: 10px; 
                    border: 1px solid #ddd; 
                    padding: 10px;
                    border-radius: 5px;
                    position: relative;
                }
                .image-container:hover { box-shadow: 0 0 10px rgba(0,0,0,0.2); }
                .point { 
                    position: absolute; 
                    width: 10px; 
                    height: 10px; 
                    border-radius: 50%; 
                    border: 1px solid black;
                    cursor: pointer;
                }
                .top-point { background-color: yellow; }
                .end1-point { background-color: blue; }
                .end2-point { background-color: red; }
                .tooltip {
                    position: absolute;
                    background: rgba(0,0,0,0.8);
                    color: white;
                    padding: 5px;
                    border-radius: 3px;
                    font-size: 12px;
                    display: none;
                    z-index: 100;
                }
                .controls {
                    margin-bottom: 20px;
                    padding: 10px;
                    background: #f5f5f5;
                    border-radius: 5px;
                }
                h1 { color: #333; }
                .image { position: relative; }
            </style>
        </head>
        <body>
            <h1>Cardiac Points Interactive Viewer</h1>
            <div class="controls">
                <p>
                    <b>Instructions:</b> Hover over points to see coordinates. 
                    Click on images to enlarge.
                </p>
                <label>
                    <input type="checkbox" id="showAxes" checked onclick="toggleAxes()">
                    Show Axes
                </label>
                <button onclick="downloadCSV()">Download Points as CSV</button>
            </div>
            
            <div class="container" id="imageContainer">
                <!-- Images will be inserted here by JavaScript -->
            </div>
            
            <div id="tooltip" class="tooltip"></div>
            
            <script>
                // Data from Python
                const pointsData = __POINTS_DATA__;
                const baseDir = "__BASE_DIR__";
                
                // Setup
                const container = document.getElementById('imageContainer');
                const tooltip = document.getElementById('tooltip');
                
                // For each timestep
                pointsData.forEach(data => {
                    // Create container for this image
                    const imageContainer = document.createElement('div');
                    imageContainer.className = 'image-container';
                    
                    // Add title
                    const title = document.createElement('h3');
                    title.textContent = `Timestep ${data.timestep}`;
                    imageContainer.appendChild(title);
                    
                    // Add image
                    const imgWrapper = document.createElement('div');
                    imgWrapper.className = 'image';
                    imgWrapper.style.width = `${data.width}px`;
                    imgWrapper.style.height = `${data.height}px`;
                    
                    const img = document.createElement('img');
                    img.src = data.image;
                    img.width = data.width;
                    img.height = data.height;
                    img.style.position = 'absolute';
                    img.style.top = '0';
                    img.style.left = '0';
                    imgWrapper.appendChild(img);
                    
                    // Add axes (initially visible)
                    const xAxis = document.createElement('div');
                    xAxis.className = 'axis x-axis';
                    xAxis.style.position = 'absolute';
                    xAxis.style.left = '0';
                    xAxis.style.top = `${data.height}px`;
                    xAxis.style.width = `${data.width}px`;
                    xAxis.style.height = '20px';
                    xAxis.style.borderTop = '1px solid rgba(0,0,0,0.5)';
                    
                    // X ticks
                    for(let x = 0; x <= data.width; x += 50) {
                        const tick = document.createElement('div');
                        tick.style.position = 'absolute';
                        tick.style.left = `${x}px`;
                        tick.style.top = '0';
                        tick.style.width = '1px';
                        tick.style.height = '5px';
                        tick.style.backgroundColor = 'rgba(0,0,0,0.5)';
                        
                        const label = document.createElement('div');
                        label.style.position = 'absolute';
                        label.style.left = `${x - 10}px`;
                        label.style.top = '6px';
                        label.style.fontSize = '10px';
                        label.textContent = x;
                        
                        xAxis.appendChild(tick);
                        xAxis.appendChild(label);
                    }
                    
                    const yAxis = document.createElement('div');
                    yAxis.className = 'axis y-axis';
                    yAxis.style.position = 'absolute';
                    yAxis.style.left = `${data.width}px`;
                    yAxis.style.top = '0';
                    yAxis.style.width = '20px';
                    yAxis.style.height = `${data.height}px`;
                    yAxis.style.borderLeft = '1px solid rgba(0,0,0,0.5)';
                    
                    // Y ticks
                    for(let y = 0; y <= data.height; y += 50) {
                        const tick = document.createElement('div');
                        tick.style.position = 'absolute';
                        tick.style.left = '0';
                        tick.style.top = `${y}px`;
                        tick.style.width = '5px';
                        tick.style.height = '1px';
                        tick.style.backgroundColor = 'rgba(0,0,0,0.5)';
                        
                        const label = document.createElement('div');
                        label.style.position = 'absolute';
                        label.style.left = '6px';
                        label.style.top = `${y - 5}px`;
                        label.style.fontSize = '10px';
                        label.textContent = y;
                        
                        yAxis.appendChild(tick);
                        yAxis.appendChild(label);
                    }
                    
                    imgWrapper.appendChild(xAxis);
                    imgWrapper.appendChild(yAxis);
                    
                    // Add points
                    // Top point
                    const topPoint = document.createElement('div');
                    topPoint.className = 'point top-point';
                    topPoint.style.left = `${data.points.top.x - 5}px`;
                    topPoint.style.top = `${data.points.top.y - 5}px`;
                    topPoint.dataset.info = `Top: (${data.points.top.x}, ${data.points.top.y})`;
                    
                    // End1 point
                    const end1Point = document.createElement('div');
                    end1Point.className = 'point end1-point';
                    end1Point.style.left = `${data.points.end1.x - 5}px`;
                    end1Point.style.top = `${data.points.end1.y - 5}px`;
                    end1Point.dataset.info = `End1: (${data.points.end1.x}, ${data.points.end1.y})`;
                    
                    // End2 point
                    const end2Point = document.createElement('div');
                    end2Point.className = 'point end2-point';
                    end2Point.style.left = `${data.points.end2.x - 5}px`;
                    end2Point.style.top = `${data.points.end2.y - 5}px`;
                    end2Point.dataset.info = `End2: (${data.points.end2.x}, ${data.points.end2.y})`;
                    
                    imgWrapper.appendChild(topPoint);
                    imgWrapper.appendChild(end1Point);
                    imgWrapper.appendChild(end2Point);
                    
                    // Setup tooltip handlers
                    [topPoint, end1Point, end2Point].forEach(point => {
                        point.addEventListener('mouseover', (e) => {
                            tooltip.textContent = e.target.dataset.info;
                            tooltip.style.display = 'block';
                            tooltip.style.left = `${e.pageX + 10}px`;
                            tooltip.style.top = `${e.pageY + 10}px`;
                        });
                        
                        point.addEventListener('mousemove', (e) => {
                            tooltip.style.left = `${e.pageX + 10}px`;
                            tooltip.style.top = `${e.pageY + 10}px`;
                        });
                        
                        point.addEventListener('mouseout', () => {
                            tooltip.style.display = 'none';
                        });
                    });
                    
                    // Add image container to wrapper
                    imageContainer.appendChild(imgWrapper);
                    
                    // Add coordinate info text below image
                    const coordInfo = document.createElement('div');
                    coordInfo.className = 'coord-info';
                    coordInfo.innerHTML = `
                        <small>
                            <b>Top:</b> (${data.points.top.x}, ${data.points.top.y})<br>
                            <b>End1:</b> (${data.points.end1.x}, ${data.points.end1.y})<br>
                            <b>End2:</b> (${data.points.end2.x}, ${data.points.end2.y})
                        </small>
                    `;
                    imageContainer.appendChild(coordInfo);
                    
                    // Add to main container
                    container.appendChild(imageContainer);
                });
                
                // Function to toggle axes visibility
                function toggleAxes() {
                    const showAxes = document.getElementById('showAxes').checked;
                    const axes = document.querySelectorAll('.axis');
                    
                    axes.forEach(axis => {
                        axis.style.display = showAxes ? 'block' : 'none';
                    });
                }
                
                // Function to download the data as CSV
                function downloadCSV() {
                    let csvContent = "Timestep,Top_X,Top_Y,End1_X,End1_Y,End2_X,End2_Y\\n";
                    
                    pointsData.forEach(data => {
                        csvContent += `${data.timestep},${data.points.top.x},${data.points.top.y},${data.points.end1.x},${data.points.end1.y},${data.points.end2.x},${data.points.end2.y}\\n`;
                    });
                    
                    const blob = new Blob([csvContent], { type: 'text/csv;charset=utf-8;' });
                    const link = document.createElement("a");
                    
                    const url = URL.createObjectURL(blob);
                    link.setAttribute("href", url);
                    link.setAttribute("download", "cardiac_points.csv");
                    link.style.visibility = 'hidden';
                    
                    document.body.appendChild(link);
                    link.click();
                    document.body.removeChild(link);
                }
                
                // Enable image click to enlarge
                document.querySelectorAll('.image-container img').forEach(img => {
                    img.addEventListener('click', (e) => {
                        if (e.target.style.transform === "scale(1.5)") {
                            e.target.style.transform = "scale(1)";
                            e.target.style.zIndex = "1";
                        } else {
                            e.target.style.transform = "scale(1.5)";
                            e.target.style.zIndex = "10";
                        }
                    });
                });
            </script>
        </body>
        </html>
        """
        
        # Create data for each timestep
        points_data = []
        
        for _, row in results_df.iterrows():
            timestep = int(row['Timestep'])
            
            # Load image
            image_path = os.path.join(input_dir, f"tst_{timestep:04d}_pred_mask_2_Skeletonized.png")
            img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
            
            if img is None:
                continue
                
            height, width = img.shape
            
            # Draw points on the image for visualization
            vis_img = np.zeros((height, width, 3), dtype=np.uint8)
            vis_img[img > 0] = [255, 255, 255]
            
            # Extract points
            point3 = (int(row['Top_X']), int(row['Top_Y']))
            end1 = (int(row['End1_X']), int(row['End1_Y']))
            end2 = (int(row['End2_X']), int(row['End2_Y']))
            
            # Draw points
            cv2.circle(vis_img, point3, 4, (0, 255, 255), -1)
            cv2.circle(vis_img, end1, 4, (255, 0, 0), -1)
            cv2.circle(vis_img, end2, 4, (0, 0, 255), -1)
            
            # Convert image to base64 for embedding in HTML
            _, buffer = cv2.imencode('.png', vis_img)
            img_str = base64.b64encode(buffer).decode('utf-8')
            
            # Add data for this timestep
            points_data.append({
                'timestep': timestep,
                'image': f'data:image/png;base64,{img_str}',
                'width': width,
                'height': height,
                'points': {
                    'top': {'x': point3[0], 'y': point3[1]},
                    'end1': {'x': end1[0], 'y': end1[1]},
                    'end2': {'x': end2[0], 'y': end2[1]}
                }
            })
        
        # Replace placeholders in the template
        import json
        html_content = html_template.replace('__POINTS_DATA__', json.dumps(points_data))
        html_content = html_content.replace('__BASE_DIR__', input_dir)
        
        # Write HTML file
        with open(output_path, 'w') as f:
            f.write(html_content)
            
        print(f"Interactive viewer saved to: {output_path}")
        
    except Exception as e:
        print(f"Error creating interactive viewer: {e}")

if __name__ == "__main__":
    # Define input and output directories
    input_dir = "/Users/ruhisharmin/Library/CloudStorage/Box-Box/aether.lab/projects/Echo.Cardio/autosegmentation/Data/CAMUS/validation_pred_mask_2_Folder_Skeletonized/patient0401"
    output_dir = "/Users/ruhisharmin/Library/CloudStorage/Box-Box/aether.lab/projects/Echo.Cardio/autosegmentation/Data/CAMUS/cardiac_point_results/patient0401"
    
    # Process the patient directory
    results = process_patient_directory(input_dir, output_dir)
    
    # Create an interactive HTML viewer
    interactive_html_path = os.path.join(output_dir, "interactive_viewer.html")
    create_interactive_viewer(results, input_dir, interactive_html_path)

## code for loop that processes all images in the patient0401 folder and saves the skeletonized versions.
- code in loop for patient no. 401 but all the timesteps inside the patient folder. 
- Then saved the skeletonized masks for each timestep in a folder named "/Users/ruhisharmin/Library/CloudStorage/Box-Box/aether.lab/projects/Echo.Cardio/autosegmentation/Data/CAMUS/validation_pred_mask_2_Folder_Skeletonized/patient0401" 
- named the files as "tst_0001_pred_mask_2_Skeletonized.png", "tst_0002_pred_mask_2_Skeletonized.png", "tst_0003_pred_mask_2_Skeletonized.png", ... etc.

Loop through all the timestep files (Ex: tst_0001_pred_mask_2.png through tst_0022_pred_mask_2.png) in the patient0401 folder
Process each image with the same steps used before in the code "skeletonize_3CornerPointDetection_fromEchoUNetoutputMASK.ipynb":
1. Convert to binary mask
2. Remove small elements using connected component analysis
3. Filter components by area (using 500 pixels as threshold)
4. Skeletonize the filtered mask
5. Save each skeletonized image to the output directory with the naming (tst_0001_pred_mask_2_Skeletonized.png, etc.)

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.morphology import skeletonize
import os

# Define input and output paths
input_dir = '/Users/ruhisharmin/Library/CloudStorage/Box-Box/aether.lab/projects/Echo.Cardio/autosegmentation/Data/CAMUS/validation_pred_mask_2_Folder/patient0401'
output_dir = '/Users/ruhisharmin/Library/CloudStorage/Box-Box/aether.lab/projects/Echo.Cardio/autosegmentation/Data/CAMUS/validation_pred_mask_2_Folder_Skeletonized/patient0401'

# Create output directory if it doesn't exist
os.makedirs(output_dir, exist_ok=True)

# Define the processing function
def process_image(input_path, output_path, min_area=500, visualize=False):
    # Load the image
    orig_mask = cv2.imread(input_path, 0)  # Read as grayscale
    
    # Check if image was loaded successfully
    if orig_mask is None:
        print(f"Error: Could not load image from {input_path}")
        return False
    
    # Convert to binary mask (threshold)
    binary_mask = (orig_mask > 127).astype(np.uint8) * 255
    
    # Remove small elements using connected component analysis
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    
    # Filter components by area
    filtered_mask = np.zeros_like(binary_mask)
    for i in range(1, num_labels):  # Skip background (0)
        if stats[i, cv2.CC_STAT_AREA] > min_area:
            filtered_mask[labels == i] = 255
    
    # Convert filtered_mask to binary for skeletonization (0s and 1s)
    binary = (filtered_mask > 0).astype(np.uint8)
    
    # Skeletonize the mask
    skeleton = skeletonize(binary).astype(np.uint8)
    
    # Multiply by 255 to get a standard binary image
    skeleton_img = skeleton * 255
    
    # Save the skeletonized image
    cv2.imwrite(output_path, skeleton_img)
    
    # Optional visualization
    if visualize:
        plt.figure(figsize=(15, 5))
        plt.subplot(1, 3, 1)
        plt.imshow(orig_mask, cmap='gray')
        plt.title('Original Mask')
        plt.axis('off')
        
        plt.subplot(1, 3, 2)
        plt.imshow(filtered_mask, cmap='gray')
        plt.title('Filtered Mask')
        plt.axis('off')
        
        plt.subplot(1, 3, 3)
        plt.imshow(skeleton_img, cmap='gray')
        plt.title('Skeletonized Mask')
        plt.axis('off')
        
        plt.tight_layout()
        plt.show()
    
    return True

# Process all timesteps for patient0401
total_processed = 0
total_files = 0

# List all files in the directory
file_list = sorted(os.listdir(input_dir))

# Filter for tst_*_pred_mask_2.png files
mask_files = [f for f in file_list if f.startswith('tst_') and f.endswith('_pred_mask_2.png')]
total_files = len(mask_files)

print(f"Found {total_files} mask files to process")

# Process each file
for mask_file in mask_files:
    # Get the timestep number (e.g., "0001" from "tst_0001_pred_mask_2.png")
    timestep = mask_file.split('_')[1]
    
    # Construct input and output paths
    input_path = os.path.join(input_dir, mask_file)
    output_filename = f"tst_{timestep}_pred_mask_2_Skeletonized.png"
    output_path = os.path.join(output_dir, output_filename)
    
    print(f"Processing {mask_file}...")
    
    # Process the image
    success = process_image(input_path, output_path, min_area=500, visualize=False)
    
    if success:
        total_processed += 1
        print(f"  Saved {output_filename}")
    else:
        print(f"  Failed to process {mask_file}")

# Print summary
print(f"\nProcessing complete: {total_processed}/{total_files} files processed successfully")
print(f"Skeletonized masks saved to: {output_dir}")

## Multi-patient Batch skeletonization (for CAMUS validation dataset)
comprehensive batch processing script that will handle all patients from 0401 through 0450 by
1. Processes multiple patient folders: The code automatically loops through each patient folder from 0401 to 0450.
2. Uses regular expressions to correctly parse the timestep numbers
Checks if directories and files exist before processing, Creates output directories as needed
3. Error handling:
handles missing patient folders, Skips files with unexpected formats, Reports errors without crashing the entire batch process
4. Progress reporting:
Shows progress for each patient, Reports success/failure counts for each patient, provides a grand total summary at the end
5. Consistent processing:
Uses the same image processing pipeline for all files, Maintains the same naming convention for output files

In [ ]:
import cv2
import numpy as np
import os
from skimage.morphology import skeletonize
import re

# Base directories
base_input_dir = '/Users/ruhisharmin/Library/CloudStorage/Box-Box/aether.lab/projects/Echo.Cardio/autosegmentation/Data/CAMUS/validation_pred_mask_2_Folder'
base_output_dir = '/Users/ruhisharmin/Library/CloudStorage/Box-Box/aether.lab/projects/Echo.Cardio/autosegmentation/Data/CAMUS/validation_pred_mask_2_Folder_Skeletonized'

# Make sure the base output directory exists
os.makedirs(base_output_dir, exist_ok=True)

# Define the processing function for a single image
def process_image(input_path, output_path, min_area=500):
    # Load the image
    orig_mask = cv2.imread(input_path, 0)  # Read as grayscale
    
    # Check if image was loaded successfully
    if orig_mask is None:
        print(f"  Error: Could not load image from {input_path}")
        return False
    
    # Convert to binary mask (threshold)
    binary_mask = (orig_mask > 127).astype(np.uint8) * 255
    
    # Remove small elements using connected component analysis
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_mask, connectivity=8)
    
    # Filter components by area
    filtered_mask = np.zeros_like(binary_mask)
    for i in range(1, num_labels):  # Skip background (0)
        if stats[i, cv2.CC_STAT_AREA] > min_area:
            filtered_mask[labels == i] = 255
    
    # Convert filtered_mask to binary for skeletonization (0s and 1s)
    binary = (filtered_mask > 0).astype(np.uint8)
    
    # Skeletonize the mask
    skeleton = skeletonize(binary).astype(np.uint8)
    
    # Multiply by 255 to get a standard binary image
    skeleton_img = skeleton * 255
    
    # Save the skeletonized image
    cv2.imwrite(output_path, skeleton_img)
    return True

# Function to process a single patient folder
def process_patient_folder(patient_number):
    patient_id = f"patient{patient_number:04d}"
    input_dir = os.path.join(base_input_dir, patient_id)
    output_dir = os.path.join(base_output_dir, patient_id)
    
    # Check if the input directory exists
    if not os.path.exists(input_dir):
        print(f"Patient directory not found: {input_dir}")
        return 0, 0
    
    # Create output directory for this patient
    os.makedirs(output_dir, exist_ok=True)
    
    # List all files in the directory
    try:
        file_list = sorted(os.listdir(input_dir))
    except Exception as e:
        print(f"Error accessing directory {input_dir}: {e}")
        return 0, 0
    
    # Filter for tst_*_pred_mask_2.png files
    mask_files = [f for f in file_list if f.startswith('tst_') and f.endswith('_pred_mask_2.png')]
    total_files = len(mask_files)
    
    if total_files == 0:
        print(f"No mask files found in {input_dir}")
        return 0, 0
    
    print(f"Found {total_files} mask files for {patient_id}")
    
    # Process each file
    processed_count = 0
    for mask_file in mask_files:
        # Get the timestep number using regex for more robustness
        match = re.search(r'tst_(\d+)_pred_mask_2\.png', mask_file)
        if not match:
            print(f"  Skipping file with unexpected format: {mask_file}")
            continue
            
        timestep = match.group(1)
        
        # Construct input and output paths
        input_path = os.path.join(input_dir, mask_file)
        output_filename = f"tst_{timestep}_pred_mask_2_Skeletonized.png"
        output_path = os.path.join(output_dir, output_filename)
        
        # Process the image
        success = process_image(input_path, output_path, min_area=500)
        
        if success:
            processed_count += 1
            print(f"  Processed {mask_file}")
    
    return processed_count, total_files

# Main processing loop for all patients
def process_all_patients(start_patient=401, end_patient=450):
    grand_total_processed = 0
    grand_total_files = 0
    
    print(f"Starting batch processing for patients {start_patient} through {end_patient}")
    print("=" * 80)
    
    for patient_num in range(start_patient, end_patient + 1):
        print(f"\nProcessing patient {patient_num:04d}...")
        processed, total = process_patient_folder(patient_num)
        
        if total > 0:
            print(f"Patient {patient_num:04d}: Processed {processed}/{total} files")
        else:
            print(f"Patient {patient_num:04d}: No files processed")
        
        grand_total_processed += processed
        grand_total_files += total
    
    print("\n" + "=" * 80)
    print(f"BATCH PROCESSING COMPLETE")
    print(f"Total processed: {grand_total_processed}/{grand_total_files} files")
    print(f"Output directory: {base_output_dir}")
    print("=" * 80)

# Run the processing for all patients
if __name__ == "__main__":
    process_all_patients(401, 450)